# RDF export of CMDI vocabularies and VLO facets in relation to concepts
Note: this notebook depends on the output of the [clarin_data](clarin_data.ipynb) notebook. The output should already be present, but if you run into problems please run this first to make sure that the data is complete and up-to-date.

In [1]:
# Preamble

# Define some common constants
CCR_API_BASE = 'https://vocabularies.clarin.eu/clavas/rest/v1/'
CCR_VOCAB = 'ccr'
VLO_API_BASE = 'https://vlo.clarin-dev.eu/api'
COMPONENT_REGISTRY_BASE = 'https://catalog.clarin.eu/ds/ComponentRegistry/rest'

# Output directories from clarin_data.ipynb to be used as input here
DATA_DIR = 'data'
CCR_DATA_DIR = DATA_DIR + '/ccr'
PROFILES_DATA_DIR = DATA_DIR + '/profiles'
VLO_FACETS_DATA_DIR = DATA_DIR + '/facets'
VLO_MAPPING_DATA_DIR = DATA_DIR + '/mappings'

import os
# Create output directories for data
RDF_DIR = 'rdf'
CCR_RDF_DIR = RDF_DIR + '/ccr'
COMPONENT_REGISTRY_RDF_DIR = RDF_DIR + '/componentRegistry'
FACETS_RDF_DIR = RDF_DIR + '/facets'
for dataDir in [RDF_DIR, CCR_RDF_DIR, COMPONENT_REGISTRY_RDF_DIR, FACETS_RDF_DIR]:
    if not os.path.exists(dataDir):
        print("Creating data directory: ", dataDir)
        os.mkdir(dataDir)

## Concept definitions form the CLARIN Concept Registry
The output of [clarin_data](clarin_data.ipynb) contains individual RDF/XML exports of the concept schemes of the CCR. Here where load these into a graph and export them into a single Turtle file.

In [2]:
from rdflib import Graph
import pprint

# Read concepts and serialize as Turtle
conceptsGraph = Graph()
for root, dirs, files in os.walk(CCR_DATA_DIR):
    for file in files:
        print('Reading RDF XML:', file)
        conceptsGraph.parse(CCR_DATA_DIR + '/' + file)
ccrRdfOutFile = CCR_RDF_DIR + '/concepts.ttl'

print('Serializing combined graph...')
conceptsGraph.serialize(destination = ccrRdfOutFile)
print('Combined graph written to', ccrRdfOutFile)


Reading RDF XML: CCR_P-LanguageResourceOntology_05399e1d-4f23-8fc1-5088-eb5afbf0cd91.xml
Reading RDF XML: CCR_P-Morphosyntax_f701a84e-9171-1e2d-22cf-6528c2571d42.xml
Reading RDF XML: CCR_P-notavailable_dc55639d-6b2b-a05a-ced0-9d394432ae43.xml
Reading RDF XML: CCR_P-Metadata_6f3f84d1-6f06-6291-4e20-4cd361cca128.xml
Reading RDF XML: CCR_P-Terminology_5b77056e-4634-1b16-a143-3697c33c17a7.xml
Reading RDF XML: CCR_P-SemanticContentRepresentation_1448f330-771b-ad11-987f-7293629f0801.xml
Reading RDF XML: CCR_P-LexicalResources_ce3edd5c-07a7-b345-dcfa-789a9b4bc980.xml
Reading RDF XML: CCR_P-SignLanguage_8f51ce1b-211d-9682-a8e7-aeb0f2a79e03.xml
Reading RDF XML: CCR_P-DialogueActs_955814a6-6c07-c143-94eb-b2551c2d51cb.xml
Reading RDF XML: CCR_P-LanguageCodes_a122c1a3-1912-fecd-07a2-8685522dfeca.xml
Reading RDF XML: CCR_P-Syntax_e7dbb854-a006-9939-4863-1236a4062a1b.xml
Reading RDF XML: CCR_P-undecided_0103ef85-e040-85f0-c3a1-7b26f5659821.xml
Reading RDF XML: CCR_P-Translation_e7ebeaa2-68f1-34c8-63

## Controlled vocabularies from CMDI component/profile specifications
An XSLT based solution to generate RDF statements for all controlled vocabularies and their items defined in the ComponentRegistry is available at [https://github.com/clarin-eric/metacat-cmd](https://github.com/clarin-eric/metacat-cmd). In the first cell below we simply parse the output of a run on all used profiles (the output of [clarin_data](clarin_data.ipynb)). 

At a later stage we may integrate/replicate the logic in this notebook.

In [3]:
### Load from pregenerated export
compRegInFile = COMPONENT_REGISTRY_RDF_DIR + '/clarin-all.ttl'
compRegOutFile = COMPONENT_REGISTRY_RDF_DIR + '/compreg-vocabs.ttl'

if not os.path.exists(compRegInFile):
    raise Exception('Input file ' + compRegInFile + 'not found. Please put this file in place to allow for processing.')

compRegGraph = Graph()
print('Parsing Component Registry vocabularies')
compRegGraph.parse(compRegInFile)

print('Serializing CompReg vocabularies graph...')
compRegGraph.serialize(destination = compRegOutFile)
print('CompReg vocabularies graph written to', compRegOutFile)

Parsing Component Registry vocabularies
Serializing CompReg vocabularies graph...
CompReg vocabularies graph written to rdf/componentRegistry/compreg-vocabs.ttl


In [4]:
#### TODO: Produce statements from the profile specifications

# Loop over data/profiles/*.xml
# Apply XSLT
# Load and store results in a single ttl file

## VLO facet definitions
For the VLO facets, we will produce two sets of statements: one for the facet values which were gathered from the VLO API and stored in [vlo-facets.json](data/facets/vlo-facets.json); and another for the facet-concept mapping (extracting information from [facetConcepts.xml](data/mappings/facetConcepts.xml).

In [5]:
from rdflib import URIRef, Namespace, Literal
VLO = Namespace('https://vlo.clarin.eu/')
VLO_FACET = Namespace('https://vlo.clarin.eu/facets/')
FACET_VALUES = Namespace('https://vlo.clarin.eu/values/')
CCR = Namespace('http://hdl.handle.net/11459/')
ISOCAT = Namespace('http://www.isocat.org/datcat/')

In [6]:
### Facets and their values

facetValuesInFile = VLO_FACETS_DATA_DIR + '/vlo-facets.json'
facetsOutFile = FACETS_RDF_DIR + '/facetValues.ttl'

from rdflib.namespace import RDF, RDFS, SKOS
vloFacetNames = []
facetValuesGraph = Graph()
facetValuesGraph.bind('vlo',VLO)
facetValuesGraph.bind('facets',VLO_FACET)
facetValuesGraph.bind('facetValues',FACET_VALUES)

import json
from hashlib import sha1
with open(facetValuesInFile, 'r') as inFile:
    data = json.load(inFile)
    for facet in data:
        facetName = facet['facet'];
        print('Facet:', facetName, '–', len(facet['values']), 'values')
        vloFacetNames += [facetName]

        # statements
        facetRef = VLO_FACET[facetName]
        schemeRef = FACET_VALUES[facetName]
        facetValuesGraph.add((facetRef, RDF.type, VLO.Facet))
        facetValuesGraph.add((facetRef, RDFS.label, Literal(facetName, lang='en')))
        facetValuesGraph.add((facetRef, VLO.facetValues, FACET_VALUES[facetName]))
        facetValuesGraph.add((schemeRef, RDF.type, SKOS.ConceptScheme))
        for value in facet['values']:
            valueRef = FACET_VALUES[facetName + '/' + sha1(bytes(value, 'utf-8')).hexdigest()]
            # value statements
            facetValuesGraph.add((valueRef, RDF.type, SKOS.Concept))
            facetValuesGraph.add((valueRef, SKOS.prefLabel, Literal(value, lang='en')))
            facetValuesGraph.add((valueRef, SKOS.inScheme, schemeRef))
            facetValuesGraph.add((valueRef, SKOS.topConceptOf, schemeRef))
            # add as top concept to scheme
            facetValuesGraph.add((schemeRef, SKOS.hasTopConcept, valueRef))

print('Serializing facet values graph...')
facetValuesGraph.serialize(destination = facetsOutFile)
print('Facet values graph written to', facetsOutFile)


Facet: languageCode – 9119 values
Facet: collection – 946 values
Facet: resourceClass – 669 values
Facet: modality – 188 values
Facet: format – 208 values
Facet: keywords – 8523 values
Facet: genre – 1456 values
Facet: subject – 100000 values
Facet: country – 336 values
Facet: organisation – 1761 values
Serializing facet values graph...
Facet values graph written to rdf/facets/facetValues.ttl


In [7]:
### Facet - concept mapping
facetConceptMappingInFile = VLO_MAPPING_DATA_DIR + '/facetConcepts.xml'
facetConceptMappingOutFile = FACETS_RDF_DIR + '/facetConcepts.ttl'

# Load the mapping file
from lxml import etree
xmlTree = etree.parse(facetConceptMappingInFile)
root = xmlTree.getroot()

facets = root.findall('facetConcept')
print('Found mapping definitions for', len(facets), 'facets')

# Construct the graph
from rdflib.namespace import RDF, DC
facetsGraph = Graph()
facetsGraph.bind('vlo',VLO)
facetsGraph.bind('facet',VLO_FACET)
facetsGraph.bind('ccr', CCR)
facetsGraph.bind('isocat',ISOCAT)

for facet in facets:
    facetName = facet.attrib['name']
    # Filter out fields that are defined in the facet concept mapping but are not used as facets in the VLO
    if facetName in vloFacetNames:
        facetRef = VLO_FACET[facetName]
        facetsGraph.add((facetRef, RDF.type, VLO.Facet))
        
        # Traverse concept mappings for this facet
        concepts = facet.findall('concept')
        print('Facet:', facetName, '-', len(concepts), 'concepts')
        for concept in concepts:
            if concept.text:
                conceptRef = URIRef(concept.text)
                facetsGraph.add((facetRef, VLO.facetConceptLink, conceptRef))
            
print('Serializing facet - concept graph...')
facetsGraph.serialize(destination = facetConceptMappingOutFile)
print('Facet - concept graph written to', facetConceptMappingOutFile)

Found mapping definitions for 28 facets
Facet: collection - 0 concepts
Facet: country - 6 concepts
Facet: languageCode - 11 concepts
Facet: organisation - 8 concepts
Facet: genre - 4 concepts
Facet: modality - 2 concepts
Facet: subject - 14 concepts
Facet: resourceClass - 9 concepts
Facet: format - 3 concepts
Facet: keywords - 4 concepts
Serializing facet - concept graph...
Facet - concept graph written to rdf/facets/facetConcepts.ttl


## Combing the graphs

In [8]:
clarinGraph = conceptsGraph + compRegGraph + facetValuesGraph + facetsGraph

In [9]:
## Serialize the combined graph

# combinedGraphOutFile = RDF_DIR + '/clarin.ttl'
# print('Serializing combined graph')
# clarinGraph.serialize(destination = combinedGraphOutFile)
# print('Combined graph written to', combinedGraphOutFile)

## Query examples

In [10]:
clarinGraph.bind('vlo',VLO)
clarinGraph.bind('ccr', CCR)
clarinGraph.bind('rdfs', RDFS)
clarinGraph.bind('cmd', Namespace('http://www.clarin.eu/cmdi#'))

In [11]:
print('All concepts mapped to a facet (querying):')

q = """
SELECT DISTINCT ?facetlabel ?concept ?conceptlabel
WHERE {
    ?facet vlo:facetConceptLink ?concept .

    ?facet a vlo:Facet ; rdfs:label ?facetlabel .    
    ?concept a skos:Concept ; skos:prefLabel ?conceptlabel .
}
ORDER BY ?facetlabel
"""

qres = clarinGraph.query(q)
for row in qres:
    print(f"Facet: {row.facetlabel} - Concept: {row.concept} '{row.conceptlabel}'")

All concepts mapped to a facet (querying):
Facet: country - Concept: http://hdl.handle.net/11459/CCR_C-3792_68c770a4-d58c-46dd-d429-5609ce5f81c3 'country name'
Facet: country - Concept: http://hdl.handle.net/11459/CCR_C-2532_d004b0a6-fd1d-3ca3-abf1-1e6aeb3e37b2 'location country'
Facet: country - Concept: http://hdl.handle.net/11459/CCR_C-2092_36cd7ca8-e412-9f29-7ea7-4a3ba4ba2c91 'country coding'
Facet: format - Concept: http://hdl.handle.net/11459/CCR_C-2571_2be2e583-e5af-34c2-3673-93359ec1f7df 'mime type'
Facet: genre - Concept: http://hdl.handle.net/11459/CCR_C-3899_c6c608e7-cb2e-1832-09ff-aee36e1f2ed4 'sub genre'
Facet: genre - Concept: http://hdl.handle.net/11459/CCR_C-2470_d191f2b2-6339-f031-b534-70d526b28357 'genre'
Facet: keywords - Concept: http://hdl.handle.net/11459/CCR_C-5436_6ab57c2c-5f8d-3561-6db6-d75da23d2637 'metadata tag'
Facet: keywords - Concept: http://hdl.handle.net/11459/CCR_C-278_336dd81a-626b-713e-c74a-34fa2ca26a71 'keyword'
Facet: languageCode - Concept: http:/